In [1]:
# -------------------------
# Standard Library Imports
# -------------------------
import copy, gc, glob, json, multiprocessing, os, random, re, shutil, sys, time, warnings, logging
from importlib import reload
from pprint import pprint

os.environ['GRISM_CAL_DIR'] = '/data/grism_cal'

# -------------------------
# Scientific Libraries
# -------------------------
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle
import numpy as np
from PIL import Image
from scipy import interpolate
from scipy.optimize import minimize

# -------------------------
# Astropy
# -------------------------
import astropy.constants as ac
import astropy.units as u
from astropy import table
from astropy.convolution import Box1DKernel, convolve
from astropy.coordinates import SkyCoord, concatenate
from astropy.io import fits, ascii
from astropy.modeling import functional_models, models
from astropy.modeling.models import Gaussian2D
from astropy.nddata import Cutout2D, NDData
from astropy.stats import sigma_clipped_stats, SigmaClip
from astropy.table import Table
from astropy.visualization.wcsaxes import WCSAxes
from astropy.wcs import WCS, FITSFixedWarning
from astropy.wcs.utils import proj_plane_pixel_scales
from astropy.visualization import astropy_mpl_style, AsinhStretch

# -------------------------
# Other Astronomy packages
# -------------------------
from photutils.detection import DAOStarFinder
from photutils.psf import EPSFBuilder, extract_stars
from astroquery.sdss import SDSS
from stpsf import NIRCam
os.environ['STPSF_PATH'] = '/home/yichenliu/stpsf-data'


# -------------------------
# Plotting and Pringting Style
# -------------------------
import scienceplots
plt.style.use(astropy_mpl_style)
plt.style.use(['bright', 'science', 'no-latex', 'notebook'])
plt.rcParams['image.origin'] = 'lower'
plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams['image.interpolation'] = 'none'
np.set_printoptions(precision=3)
np.set_printoptions(suppress=True)

# -------------------------
# Suppress Astropy Warnings
# -------------------------
warnings.simplefilter('ignore', FITSFixedWarning)

# -------------------------
# Jupyter Display Settings
# -------------------------
%matplotlib inline
%config InlineBackend.figure_format = 'retina'


# -------------------------
# GPU/ML related
# -------------------------
import torch
from torch.autograd import grad

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


/home/aakavan/miniconda3/envs/aas_dingo/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


In [2]:
from dingo import grism, utils, galaxy, kinematics, fitting

In [3]:
LOG = logging.getLogger(__name__)
LOG.setLevel(logging.INFO)
LOG.handlers.clear()

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(name)s - %(message)s',
    datefmt='%H:%M:%S'
)

In [4]:
def recursive_reload(module, visited=None):
    import importlib, types
    if visited is None:
        visited = set()
    if module in visited:
        return
    visited.add(module)
    for attr in dir(module):
        sub = getattr(module, attr)
        if isinstance(sub, types.ModuleType) and sub.__name__.startswith(module.__name__):
            recursive_reload(sub, visited)
    importlib.reload(module)


In [5]:
def asinhstretch(im): return np.arcsinh(im/sigma_clipped_stats(im)[2])

## select sources

In [6]:
TBL_PHOT = Table.read('/data/backup_home/aakavan/aakavan/edr_products/sapphires_edr_phot_cat.fits', hdu=1)
TBL_SPEC = Table.read('/data/backup_home/aakavan/aakavan/edr_products/sapphires_edr_spec_cat.fits', hdu=1)

In [16]:
from astropy.table import Table
import numpy as np

# Assume TBL_SPEC is already loaded as an Astropy Table

# line = 'PaA'
line = 'Ha'

# Output lists
ids, ras, decs, zs, paA_waves = [], [], [], [], []

for row in TBL_SPEC:
    line_names = row['name'].split('|')
    if line in line_names:
        idx = line_names.index(line)
        wave_values = list(map(float, row['wave'].split('|')))
        paA_wave = wave_values[idx]

        ids.append(row['ID'])
        ras.append(row['RA'])
        decs.append(row['DEC'])
        zs.append(row['zspec'])
        paA_waves.append(paA_wave)

# Create the new table
paA_table = Table(
    [ids, ras, decs, zs, paA_waves],
    names=['ID', 'RA', 'DEC', 'z', 'wave']
)

paA_table

ID,RA,DEC,z,wave
int64,float64,float64,float64,float64
491,63.95359,-24.19675,4.5469,3.641
727,63.95782,-24.19529,4.4614,3.585
789,63.9596,-24.19509,3.9512,3.25
977,63.95019,-24.19384,5.5749,4.315
1097,63.94569,-24.19258,6.561,4.961
1449,63.96689,-24.19149,5.9187,4.543
2074,63.96409,-24.189,3.9242,3.233
2240,63.9609,-24.1883,6.2733,4.776
2252,63.95454,-24.18753,3.8811,3.205


In [21]:
from astropy.nddata import Cutout2D
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.io import fits
from astropy.wcs import WCS
import numpy as np
import glob
import os
from tqdm import tqdm

# Constants
GRISM_DIR = '/data/sapphires/grism/'
IMAGE_DIR = '/data/sapphires/all_mosaics/YF_working/direct_imaging'
# FILTER = 'F356W'
FILTER = 'F444W'
fit_r = 30

# Load grism models
WRANGE_R, spatial_model_R, disp_model_R = grism.load_nircam_wfss_model(pupil='R', module='A', filter=FILTER)
WRANGE_C, spatial_model_C, disp_model_C = grism.load_nircam_wfss_model(pupil='C', module='A', filter=FILTER)

# Discover data paths
grism_paths = glob.glob(os.path.join(GRISM_DIR, f'jw064341[1-3]*nrc[a-b]long_rate_lv1.5.fits'))
image_paths = glob.glob(os.path.join(IMAGE_DIR, 'obsnum1[1-3]*', f'{FILTER}_imaging', '*_crf.fits'))

# Helpers
def get_cutout_from_paths(paths, coord, size):
    cutouts = []
    for path in paths:
        with fits.open(path) as hdu:
            data = hdu['SCI'].data
            wcs = WCS(hdu['SCI'].header)
        if not wcs.footprint_contains(coord):
            continue
        cutout = Cutout2D(data, coord, size, wcs, mode='partial', fill_value=np.nan)
        cutouts.append(np.array(cutout.data, dtype=np.float32))
    return cutouts

def collect_grism_cutouts(coord, wave_obs, pupil):
    cutouts = []
    for path in grism_paths:
        with fits.open(path) as hdu:
            head = hdu[0].header
            if head['FILTER'] != FILTER or head['PUPIL'][-1] != pupil:
                continue
            wcs = WCS(hdu['SCI'].header)
            if not wcs.footprint_contains(coord):
                continue
            WRANGE, spat_model, disp_model = grism.load_nircam_wfss_model(
                pupil=head['PUPIL'][-1], module=head['MODULE'], filter=head['FILTER']
            )
            if not WRANGE[0] <= wave_obs <= WRANGE[1]:
                continue
            px, py = wcs.world_to_pixel(coord)
            x, y = grism.find_pixel_location(WRANGE, spat_model, disp_model, pupil, px, py, wave_obs)
            if not (0 < x < 2048 and 0 < y < 2048):
                continue
            data = hdu['EMLINE'].data
            cutout = Cutout2D(data, (x, y), fit_r*2+1, mode='partial', fill_value=np.nan)
            cutouts.append(np.array(cutout.data, dtype=np.float32))
    return cutouts

# Outputs
direct_cutouts_list = []
grismR_cutouts_list = []
grismC_cutouts_list = []
filtered_rows = []

# Main loop
for row in tqdm(paA_table):
    ID, ra, dec, wave_obs = row['ID'], row['RA'], row['DEC'], row['wave']
    coord = SkyCoord(ra*u.deg, dec*u.deg)

    # Collect cutouts
    image_cutouts = get_cutout_from_paths(image_paths, coord, fit_r*2+1)
    grism_R_cutouts = collect_grism_cutouts(coord, wave_obs, pupil='R')
    grism_C_cutouts = collect_grism_cutouts(coord, wave_obs, pupil='C')

    if not grism_R_cutouts or not grism_C_cutouts:
        continue  # skip sources without both R and C grism data

    coadd_image = utils.coadd_images(image_cutouts) if image_cutouts else np.full((fit_r*2+1, fit_r*2+1), np.nan)
    coadd_R = utils.coadd_images(grism_R_cutouts)
    coadd_C = utils.coadd_images(grism_C_cutouts)

    # Store valid results
    direct_cutouts_list.append(coadd_image)
    grismR_cutouts_list.append(coadd_R)
    grismC_cutouts_list.append(coadd_C)
    filtered_rows.append(row)

# Convert to filtered table
from astropy.table import Table
paA_table_filtered = Table(rows=filtered_rows, names=paA_table.colnames)


100%|██████████| 332/332 [04:49<00:00,  1.15it/s]


In [22]:
import matplotlib.pyplot as plt
import numpy as np
from astropy.cosmology import Planck15 as cosmo
import matplotlib.patches as patches
from astropy.stats import sigma_clipped_stats
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
import os


def asinhstretch(im):
    return np.arcsinh(im / sigma_clipped_stats(im)[2])


def save_cutout_panels(paA_table_filtered,
                        direct_cutouts_list,
                        grismR_cutouts_list,
                        grismC_cutouts_list):

    for idx, row in enumerate(paA_table_filtered):

        fig, axes = plt.subplots(1, 3, figsize=(9,3))

        ID, z = row['ID'], row['z']

        coadd_image = direct_cutouts_list[idx]
        coadd_R = grismR_cutouts_list[idx]
        coadd_C = grismC_cutouts_list[idx]

        cutouts = [
            asinhstretch(coadd_image),
            coadd_R,
            coadd_C
        ]

        titles = [
            'Image',
            'Grism R',
            'Grism C'
        ]

        for j in range(3):

            ax = axes[j]

            ax.imshow(
                cutouts[j],
                origin='lower',
                cmap='gist_heat'
            )

            ax.set_xticks([])
            ax.set_yticks([])

            ax.grid(False)

            for spine in ax.spines.values():
                spine.set_edgecolor('black')
                spine.set_linewidth(0.8)

            ax.set_title(titles[j], fontsize=10)

            if j == 0:

                ax.text(
                    0.03,
                    0.93,
                    f'ID: {ID}\nz: {z:.3f}',
                    transform=ax.transAxes,
                    fontsize=9,
                    ha='left',
                    va='top',
                    c='white'
                )

                ang_diam_dist = cosmo.angular_diameter_distance(z).to('kpc')

                kpc_per_arcsec = ang_diam_dist.value / 206265

                pixel_scale = 0.031

                pixels_per_kpc = 1 / (kpc_per_arcsec * pixel_scale)

                bar_length_pix = int(10 * pixels_per_kpc)

                y_bar = coadd_image.shape[0] * 0.05

                x_bar_start = coadd_image.shape[1] * 0.05

                rect = patches.Rectangle(
                    (x_bar_start, y_bar),
                    bar_length_pix,
                    2,
                    color='white'
                )

                ax.add_patch(rect)

                ax.text(
                    x_bar_start + bar_length_pix / 2,
                    y_bar + 5,
                    '10 kpc',
                    color='white',
                    ha='center',
                    va='bottom',
                    fontsize=8
                )

        plt.tight_layout()
        
        os.makedirs("all_sources_grism_imaging", exist_ok=True)
        plt.savefig(
            f"all_sources_grism_imaging/{ID}_{FILTER}_{line}_grism_overview.png",
            dpi=300,
            bbox_inches='tight'
        )
        plt.close()

save_cutout_panels(
    paA_table_filtered,
    direct_cutouts_list,
    grismR_cutouts_list,
    grismC_cutouts_list
)

In [23]:
len(paA_table_filtered)

88

In [ ]:
import logging

logger = logging.getLogger('astroquery')
logger.handlers.clear()  # removes all handlers

# Or more carefully:
for handler in logger.handlers[:]:
    logger.removeHandler(handler)


# # this will override the setting above and apply to all the loggers
# logging.basicConfig(
#     level=logging.INFO,
#     format='%(asctime)s - %(levelname)s - %(name)s - %(message)s',
#     datefmt='%H:%M:%S'
# )